# 196. Delete Duplicate Emails

**Difficulty:** Easy &nbsp;|&nbsp; **Topics:** database, delete, subquery
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/delete-duplicate-emails/)

```
Table: Person
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| email       | varchar |
+-------------+---------+
id is the primary key. Each row contains an email. The emails will not contain
uppercase letters.
```

Write a solution to **delete** all duplicate emails, keeping only one unique email with
the **smallest** `id`.

For SQL users, please note that you are supposed to write a `DELETE` statement and not
a `SELECT` one. **The final order of the table does not matter.**

---

### Example

```
Person before:                   Person after:
+----+------------------+        +----+------------------+
| id | email            |        | id | email            |
+----+------------------+        +----+------------------+
| 1  | john@example.com |        | 1  | john@example.com |
| 2  | bob@example.com  |        | 2  | bob@example.com  |
| 3  | john@example.com |        +----+------------------+
+----+------------------+
```

`john@example.com` is repeated, so the row with the larger id (3) is deleted.

---

The first problem in this folder that **changes** the data instead of reporting it -
so the harness has to read the table back afterwards, exactly like #707's harness had
to read the linked list back. It also contains a restriction MySQL has and SQLite does
not, which is worth meeting deliberately rather than in an exam.

## Before you write anything

**1.** Start from #182. You already know how to find the duplicated emails. Now you need
something slightly different: for each email, the `id` you are going to **keep**. Write
the `SELECT` that produces one row per distinct email with the smallest id, and run it
with `show`. That result is the entire plan.

**2.** So the deletion is "delete every row whose id is not in that keep-list". Write it
as a `DELETE ... WHERE id NOT IN (...)`. Then stop and check it against the trap from
#183: is the subquery's column nullable here? Say why you are safe this time - the
answer is about `MIN(id)` and the primary key, and being able to say *why* a dangerous
construct is safe in this case is better than avoiding it superstitiously.

**3.** **Run it in SQLite - it works. It will not work on LeetCode.** MySQL refuses to
let a `DELETE` read from the same table in a subquery:

```
ERROR 1093 (HY000): You can't specify target table 'Person' for update in FROM clause
```

The fix is a wrapper that forces MySQL to materialise the subquery first:

```sql
DELETE FROM Person WHERE id NOT IN (
    SELECT keep FROM (SELECT MIN(id) AS keep FROM Person GROUP BY email) AS t
)
```

Write down what the extra `SELECT ... FROM (...) AS t` actually changes. Note that it is
a restriction of one engine and not of SQL - a useful thing to know exists, and a useful
reminder that "it works on my machine" has a specific meaning here.

**4.** The other standard form is a **self join**:

```sql
DELETE p1 FROM Person p1 JOIN Person p2 ON p1.email = p2.email AND p1.id > p2.id
```

That is MySQL's multi-table delete syntax and SQLite does not have it. Read it and say,
in one sentence, why `p1.id > p2.id` keeps exactly the smallest id of each group - and
what would happen if you wrote `<>` instead.

**5.** What must happen to an email that appears exactly **once**? And to a table with
no duplicates at all? Your statement must be a no-op in both cases. Which part of your
`WHERE` guarantees it?

**6.** How do you test a `DELETE`? It returns no rows. So what does the harness have to
do after running your statement, and what would it have to do to catch a statement that
deleted **everything**? (Predict it, then read the harness - it is a different shape from
every other one in this folder.)

## Two routes

**A - delete everything that is not a minimum** *(write this first, portable)*

```sql
DELETE FROM Person
WHERE id NOT IN (
    SELECT keep FROM (SELECT MIN(id) AS keep FROM Person GROUP BY email) AS t
);
```

The inner query is one row per distinct email holding the id to keep - #182's grouping
with `MIN` instead of `COUNT`. Everything else goes. The middle `SELECT ... AS t` is the
MySQL wrapper from question 3; it is harmless in SQLite, so writing it here means one
statement that runs in both places.

If an email appears once, its only id **is** the minimum, so it is in the keep-list and
survives. That is question 5's answer, and it is why no special case is needed.

**B - the correlated form**

```sql
DELETE FROM Person
WHERE id > (SELECT MIN(id) FROM Person p2 WHERE p2.email = Person.email);
```

Reads as "delete me if there is an earlier row with my email". Also correct, also runs
in SQLite, and also blocked by MySQL's rule - which is a good demonstration that the
restriction is about the *engine*, not about which shape you chose.

> **A `DELETE` is a `SELECT` you cannot undo.** The habit worth building, before you ever
> run one against a table that matters: write it as `SELECT * FROM ... WHERE <same
> condition>` first, look at the rows, and only then change the first word. Every one of
> the test cases below is a version of "did you delete more than you meant to".

In [ ]:
SOLUTION = '''
'''

### The test harness

Question 6's answer, and it is the only harness in this folder shaped like this.

A `DELETE` returns no rows, so there is nothing to compare against. Instead `check`
builds the table, runs your statement, and then **reads the whole table back** with
`SELECT id, email FROM Person ORDER BY id` - the same trick #707's harness used for a
class whose methods all returned `None`.

On failure it prints the table before, the table after, and what it should have been,
and then splits the difference into the two mistakes that matter and are easy to
confuse: rows you **deleted that you should have kept**, and duplicate rows you
**left behind**. A statement that deletes the entire table and one that deletes nothing
both fail here, loudly and differently.

Note that every case gets a **fresh** database, so your statement never runs twice
against the same rows.

> **SQLite here, MySQL on LeetCode.** SQLite lets a `DELETE` read the table it is
> deleting from; MySQL does not (error 1093). Write the wrapped form from route A and the
> same statement passes in both. See question 3.

Run this cell; don't edit it.

In [ ]:
import sqlite3

SCHEMA = """CREATE TABLE Person (id INTEGER, email TEXT);"""

READ_BACK = "SELECT id, email FROM Person ORDER BY id"


def check(name, data_sql, expected):
    """Build a fresh database, run SOLUTION as a statement, then read the table back."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(SCHEMA)
        if data_sql.strip():
            con.executescript(data_sql)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       the harness could not build the tables: {e}")
        return False

    before = [tuple(r) for r in con.execute(READ_BACK).fetchall()]

    if not SOLUTION.strip():
        print(f"FAIL {name}")
        print("       SOLUTION is empty - write your statement in the cell above")
        return False

    try:
        con.executescript(SOLUTION)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       your statement raised {type(e).__name__}: {e}")
        return False

    got = [tuple(r) for r in con.execute(READ_BACK).fetchall()]
    if got == expected:
        print(f"OK   {name}")
        return True

    print(f"FAIL {name}")
    print(f"       table before  {before}")
    print(f"       table after   {got}")
    print(f"       should be     {expected}")
    deleted_too_much = [r for r in expected if r not in got]
    left_behind = [r for r in got if r not in expected]
    if deleted_too_much:
        print(f"       you DELETED rows you should have kept: {deleted_too_much}")
    if left_behind:
        print(f"       you LEFT duplicates behind: {left_behind}")
    return False


def show(name, data_sql, query):
    """Run any query against a dataset and print it - for exploring, not for grading."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    if data_sql.strip():
        con.executescript(data_sql)
    cur = con.execute(query)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"-- {name}")
    print("   " + " | ".join(str(c) for c in cols))
    for r in rows:
        print("   " + " | ".join("NULL" if v is None else str(v) for v in r))
    if not rows:
        print("   (no rows)")

In [ ]:
# tests
check("the LeetCode example", '''
INSERT INTO Person VALUES (1,'john@example.com'), (2,'bob@example.com'),
                          (3,'john@example.com');
''', [(1,'john@example.com'), (2,'bob@example.com')])

check("question 5: no duplicates - nothing may be deleted", '''
INSERT INTO Person VALUES (1,'a@x.com'), (2,'b@x.com'), (3,'c@x.com');
''', [(1,'a@x.com'), (2,'b@x.com'), (3,'c@x.com')])

check("a single row", '''
INSERT INTO Person VALUES (7,'only@one.com');
''', [(7,'only@one.com')])

check("an empty table", '', [])

check("every row is the same email - one survives", '''
INSERT INTO Person VALUES (1,'x@y.com'), (2,'x@y.com'), (3,'x@y.com'), (4,'x@y.com');
''', [(1,'x@y.com')])

check("the smallest id is not the first row inserted", '''
INSERT INTO Person VALUES (9,'a@x.com'), (3,'a@x.com'), (5,'a@x.com');
''', [(3,'a@x.com')])

check("two separate groups of duplicates", '''
INSERT INTO Person VALUES (1,'a@x.com'), (2,'b@x.com'), (3,'a@x.com'),
                          (4,'b@x.com'), (5,'a@x.com');
''', [(1,'a@x.com'), (2,'b@x.com')])

check("duplicates interleaved with unique addresses", '''
INSERT INTO Person VALUES (1,'a@x.com'), (2,'unique1@x.com'), (3,'a@x.com'),
                          (4,'unique2@x.com'), (5,'b@x.com'), (6,'b@x.com');
''', [(1,'a@x.com'), (2,'unique1@x.com'), (4,'unique2@x.com'), (5,'b@x.com')])

check("ids that are large and unsorted", '''
INSERT INTO Person VALUES (1000,'z@z.com'), (7,'z@z.com'), (42,'q@q.com'), (41,'q@q.com');
''', [(7,'z@z.com'), (41,'q@q.com')])

check("similar addresses are not duplicates", '''
INSERT INTO Person VALUES (1,'a@b.com'), (2,'a@b.co'), (3,'ab@b.com');
''', [(1,'a@b.com'), (2,'a@b.co'), (3,'ab@b.com')])

## After it passes

- **Do the safety drill.** Take your finished statement, change `DELETE` to
  `SELECT * `, and run it with `show` on the interleaved dataset. Look at the rows it
  lists - those are the ones you were about to destroy. Make this reflex automatic now,
  while the table is imaginary.
- **Break it in the two directions.** Change `NOT IN` to `IN` and run the tests: you have
  kept exactly the rows you meant to delete. Then change `MIN(id)` to `MAX(id)`: you keep
  the *last* of each group instead of the first. Both fail, and the harness tells you
  which of the two mistakes you made, which is the whole reason it prints them
  separately.
- **Write the MySQL self-join version** from question 4 as a comment in your solution
  cell. You cannot run it here, but you will meet it in other people's code and it is
  worth being able to read.
- **Then ask the question the problem avoided.** Real deduplication almost never keeps
  "the smallest id" - it keeps the most recently updated row, or merges the two records.
  Add an `updated_at` column to the schema and rewrite the keep-list to pick the newest
  per email. Suddenly ties are possible and you need a tie-breaker; work out what it
  should be.
- Siblings: **#182 Duplicate Emails** (the `SELECT` this deletes from - do it first if
  you have not), #1667 Fix Names in a Table and #627 Swap Salary (the other two `UPDATE`
  problems, and the same read-the-table-back harness shape).